# CaseTrail run 2: teacher and students on selected chunks

Run 1 (`train_retriever.ipynb`) trained the teacher on 3 x 3 **random** chunk pairs; its loss stayed at ln 2 = 0.693
(chance), because random parts of two long judgments rarely contain the point that links them. Here the teacher sees
the **4 most similar chunk pairs**, picked with the zero-shot e5-small encoder, with a learning-rate warm-up.
It also prints a **dev pair accuracy** right after training: about 0.5 means it still learned nothing.

Run 1's plain student also got worse than zero-shot after one epoch (dev 0.3526 -> 0.3391): it was trained on 4 random
chunks per candidate but evaluated with MaxSim over all chunks. Here both students get, for each candidate, the 4
chunks that best match the query's chunks (picked with the same zero-shot e5).

**Before running:** rebuild the code zip locally (`python scripts/make_kaggle_bundle.py`) and upload it as a
**new version** of the `casetrail-code` dataset. Same settings as run 1: GPU T4, Internet on, `HF_TOKEN` secret.

**Comes back:** `casetrail_outputs_sel.zip` (students `plain_sel` and `distilled_sel`, teacher re-ranking runs `_sel`, training logs).
Every output name ends in `_sel`, so nothing from run 1 is overwritten.

In [ ]:
import os, shutil, glob, zipfile
# Find our code wherever Kaggle mounted the dataset (as the zip, or already unzipped)
zips = glob.glob('/kaggle/input/**/casetrail_code.zip', recursive=True)
trees = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)]
assert zips or trees, 'Dataset casetrail-code not found: add it with "Add Input" (right panel)'
os.makedirs('/kaggle/working/casetrail', exist_ok=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/casetrail')
else:
    shutil.copytree(trees[0], '/kaggle/working/casetrail', dirs_exist_ok=True)
os.chdir('/kaggle/working/casetrail')
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['PYTHONUTF8'] = '1'

## 1. Data, BM25 index, training data (same as run 1; the chunks use the same MiniLM tokenizer)

In [ ]:
!python scripts/download_data.py
!python scripts/build_index.py --stem on --ngrams 2 --no-passages
!python scripts/download_models.py --models sentence-transformers/all-MiniLM-L6-v2 intfloat/e5-small-v2
!python scripts/prepare_training_data.py --tokenizer models/hf/sentence-transformers__all-MiniLM-L6-v2

## 2. Teacher on selected chunk pairs
Batch 4 judgment pairs x 4 chunk pairs = 16 sequences per step (run 1 used 18 without running out of memory).

In [ ]:
TEACHER = 'BAAI/bge-reranker-base'
SELECTOR = 'models/hf/intfloat__e5-small-v2'   # the student chosen on dev in run 1 (zero-shot 0.3526 vs 0.3357)
!python scripts/train_teacher.py --model {TEACHER} --epochs 1 --batch-size 4 --select-with {SELECTOR} --tag sel

## 3. Students on selected chunks: plain (contrastive only) and distilled from the new teacher


In [ ]:
!python scripts/train_student.py --model {SELECTOR} --variant plain_sel --select-with {SELECTOR} --epochs 3 --batch-size 8
!python scripts/train_student.py --model {SELECTOR} --variant distilled_sel --distill --teacher-scores teacher_scores_sel.json --select-with {SELECTOR} --epochs 3 --batch-size 8

## 4. Package the outputs (download `casetrail_outputs_sel.zip` from the Output panel)

In [ ]:
keep = glob.glob('models/student_plain_sel/*') + glob.glob('models/student_distilled_sel/*') + glob.glob('indexes/runs/pcr_*_teacher_rerank_sel.json')      + glob.glob('results/training_teacher_sel.json') + glob.glob('results/training_student_*_sel.json')
with zipfile.ZipFile('/kaggle/working/casetrail_outputs_sel.zip', 'w', zipfile.ZIP_DEFLATED) as out:
    for path in keep:
        out.write(path, path)
print(len(keep), 'files ->', os.path.getsize('/kaggle/working/casetrail_outputs_sel.zip') // 1_000_000, 'MB')